# OpenQuant Quickstart

5 分钟跑通：**数据 → 因子 → IC → 回测 → 报告**。

本 notebook 用 `MockSource` 合成数据，**不需要 Tushare token / 不需要 DeepSeek key / 不连任何外部 API**。
想看真实 A 股回测，参考 `RESULTS.md` 或运行 `scripts/sync_*.py` 自己拉数据。

## 环境准备

```bash
uv venv -p 3.11 && source .venv/bin/activate
uv pip install -e ".[dev]"
uv pip install jupyterlab && jupyter lab notebooks/
```

## 1. 因子库速览

OpenQuant 自带 122 个因子，3 大来源：

In [ ]:
from open_quant.factors import default_engine

eng = default_engine()
all_factors = eng.names()
print(f'总共 {len(all_factors)} 个因子')

# 按前缀分类
categories = {
    'baseline (10)': [n for n in all_factors if not n.startswith(('alpha', 'gtja', 'ml_'))],
    'Alpha101 (~57)': [n for n in all_factors if n.startswith('alpha')],
    'GTJA Alpha191 (~55)': [n for n in all_factors if n.startswith('gtja')],
    'ML composite': [n for n in all_factors if n.startswith('ml_')],
}
for cat, items in categories.items():
    print(f'\n{cat}: {len(items)} 个')
    print(f'  示例: {items[:6]}')

## 2. 合成数据 (MockSource)

20 只虚拟股票 × 2 年 ≈ 9700 行 OHLCV。所有 A 股规则（T+1、涨跌停、佣金、印花税）都跑得通。

In [ ]:
import polars as pl
from datetime import date
from open_quant.data.sources import MockSource

symbols = [f'60000{i}.SH' for i in range(10)] + [f'00000{i}.SZ' for i in range(10)]
src = MockSource(symbols=symbols, seed=42)

daily = src.daily(None, date(2022, 1, 1), date(2023, 12, 31)).with_columns(
    pl.col('trade_date').str.strptime(pl.Date, '%Y%m%d', strict=False)
).rename({'ts_code': 'symbol', 'vol': 'volume'})

basic = src.daily_basic(None, date(2022, 1, 1), date(2023, 12, 31)).with_columns(
    pl.col('trade_date').str.strptime(pl.Date, '%Y%m%d', strict=False)
).rename({'ts_code': 'symbol'})

panel = daily.join(basic, on=['symbol', 'trade_date'], how='left').with_columns([
    pl.lit('sse_main').alias('board'),
    pl.lit(False).alias('is_st'),
    pl.lit(False).alias('suspended'),
])
print(f'panel: {panel.height} rows, {panel["symbol"].n_unique()} symbols')
panel.head(3)

## 3. 计算一个因子 + 看 IC

以 `mom_20d` (20日动量) 为例。注意合成数据没有真信号，所以 IC 应该接近 0 — 这正是预期。

In [ ]:
from open_quant.factors import evaluate_factor

result = eng.compute('mom_20d', panel)
print(f'factor: {result.name}, {result.data.height} (symbol, date) 点')
result.data.head(3)

In [ ]:
ev = evaluate_factor(result.data, panel, name='mom_20d', primary_horizon=5)
print('=== mom_20d 5日远期 IC 统计 ===')
for k, v in ev.summary().items():
    print(f'  {k:12s}: {v:+.4f}')
print('\n合成数据上 IC 接近 0 是预期的 — 真实 A 股数据上 vol_20d/alpha042 等能达到 ICIR > 3')

## 4. 跑一个简单回测

策略：每周五调仓，按 `vol_20d` (低波动) 选 top 5，等权。

In [ ]:
from open_quant.backtest import EventBacktester, BacktestConfig
from open_quant.strategies import FactorWeight, MultiFactorStrategy

strat = MultiFactorStrategy(
    factors=[FactorWeight(name='vol_20d', weight=1.0, direction=1)],
    top_n=5, rebalance_freq='W-FRI', max_weight=0.20,
)

bt = EventBacktester(BacktestConfig(
    start=date(2022, 6, 1), end=date(2023, 12, 31),
    initial_cash=1_000_000,
))

res = bt.run(panel.filter(pl.col('trade_date') >= date(2022, 2, 1)), strat)
print('=== 回测结果 ===')
for k, v in res.summary().items():
    print(f'  {k:15s}: {v:+.4f}')

## 5. 看 NAV 曲线

前 30 天的净值变化：

In [ ]:
res.nav.head(30)

## 6. 生成 HTML 详细报告

包含 KPI 卡片 / SVG 累计收益曲线 / 回撤曲线 / 月度收益表 / Top winners&losers。

In [ ]:
from open_quant.monitor import daily_report
from pathlib import Path

nav_for_report = [
    {**r, 'nav': float(r['nav']), 'cash': float(r['cash']),
     'market_value': float(r['market_value']), 'daily_ret': float(r.get('daily_ret', 0) or 0)}
    for r in res.nav.to_dicts()
]
fills_for_report = [
    {'trade_date': f.trade_date, 'symbol': f.symbol, 'side': f.side,
     'qty': f.qty, 'price': f.price, 'cost': f.cost}
    for f in res.fills
]

out_path = Path('quickstart_report.html')
daily_report(
    nav=nav_for_report, fills=fills_for_report,
    strategy='quickstart_demo', out_path=out_path,
    initial_cash=1_000_000,
)
print(f'报告已生成: {out_path.resolve()}')
print(f'在浏览器中打开: open {out_path.resolve()}')

## 7. (选读) Agent Overlay 开关

如果你配了 `configs/data_sources.yaml` 的 DeepSeek API key，可以这样开启 LLM 二审：

```yaml
# 在 strategy yaml 中加这一段
qualitative_overlay:
  enabled: true             # 关闭只需 false / 删这段
  agents:
    fundamentals: true
    news: true
    technical: true
  pre_filter:
    only_risky: true        # 蓝筹跳过 LLM，实测省 ~40-50% 成本
  decision:
    veto_threshold: 0.85
```

然后 CLI：

```bash
open-quant agents config              # 查看资源配置
open-quant agents test 600519.SH      # 单股 4-agent 评估
open-quant agents eval --from ... --to ...   # A/B 对比量化与量化+LLM
```

实测在 2 周/10 候选股样本上 LLM overlay 拖累 -1.1pp（见 `RESULTS.md`），主要被稀疏的 AkShare 新闻拖累。等 Tushare 升级或加财联社爬虫后再启用。

## 下一步

- 拉真实 A 股数据: `python scripts/sync_hs300_top50.py`
- 训练 ML 复合因子: `python scripts/train_ml_composite.py`
- 严格 OOS 验证: `python scripts/train_strict_holdout.py`
- 跑 paper trading: `python scripts/paper_daily.py --config ... --from ... --to ...`
- 完整结果与方法学: 看仓库根的 `RESULTS.md`